# Repository Recommender - Data Exploration & Baseline Model

This notebook demonstrates the DS workflow for the git-query recommender:
1. Load repository dataset from the gateway API
2. Explore data distributions
3. Extract features for ranking
4. Train a baseline ranking model
5. Evaluate with IR metrics

In [1]:
import sys
sys.path.insert(0, '../../..')  # Add project root to path

import os
from dotenv import load_dotenv
load_dotenv(dotenv_path='../../../.env')  # Load .env from project root

from src.recommender.data import RepoDataset, FeatureExtractor
import pandas as pd
import numpy as np
import fastparquet
import pyarrow

/Users/vladul/Code/chatbots/git-query/.venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_path" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ('settings_',)`.
  warnings.warn(
/Users/vladul/Code/chatbots/git-query/.venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/vladul/Code/chatbots/git-query/.venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_type" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [ ]:
# Fetch from gateway (first time)
ds = RepoDataset.from_gateway(
    url=os.getenv("API_BASE_URL"),
    api_key=os.getenv("APIKEY_MONGODB"),
    max_repos=1000 # change the amount of repos you want to pull from db set to None for the whole dataset
 )
ds.save("data/repos.parquet")

# Load from local cache (faster)
#ds = RepoDataset.from_local("data/repos_1000.parquet")
#print(f"Loaded {len(ds)} repositories")
#ds.summary()

[RepoDataset] Count API returned 1 -- using fallback (will stop on empty batch).
[RepoDataset] Target repo count: 100
[RepoDataset] Fetched batch 1 (100 docs, total so far: 100)
[RepoDataset] Done. 100 repositories loaded.
[RepoDataset] Saved 100 repos to data/repos_1000.parquet


In [3]:
df = ds.to_dataframe()
df.head()

,_id,owner,name,stars,forks,watchers,isFork,isArchived,languages,languageCount,...,parent,full_name,repo_id,language,is_fork,is_archived,created_at,pushed_at,topics_list,topics_str
0,freeCodeCamp/freeCodeCamp,freeCodeCamp,freeCodeCamp,426893,41377,8588,False,False,"[{'name': 'TypeScript', 'size': 2843131}, {'na...",6,...,None,freeCodeCamp/freeCodeCamp,freeCodeCamp/freeCodeCamp,TypeScript,False,False,2014-12-24 17:49:19+00:00,2025-08-31 09:33:14+00:00,"[{'name': 'learn-to-code', 'stars': 480}, {'na...","{'name': 'learn-to-code', 'stars': 480},{'name..."
1,codecrafters-io/build-your-own-x,codecrafters-io,build-your-own-x,415801,38976,6247,False,False,"[{'name': 'Markdown', 'size': 46108}]",1,...,None,codecrafters-io/build-your-own-x,codecrafters-io/build-your-own-x,Markdown,False,False,2018-05-09 12:03:18+00:00,2025-08-29 00:08:20+00:00,"[{'name': 'programming', 'stars': 1849}, {'nam...","{'name': 'programming', 'stars': 1849},{'name'..."
2,sindresorhus/awesome,sindresorhus,awesome,396445,31414,8011,False,False,[],0,...,None,sindresorhus/awesome,sindresorhus/awesome,NaN,False,False,2014-07-11 13:42:37+00:00,2025-07-18 18:37:33+00:00,"[{'name': 'awesome', 'stars': 85421}, {'name':...","{'name': 'awesome', 'stars': 85421},{'name': '..."
3,EbookFoundation/free-programming-books,EbookFoundation,free-programming-books,366964,64025,9879,False,False,"[{'name': 'Python', 'size': 26631}, {'name': '...",2,...,None,EbookFoundation/free-programming-books,EbookFoundation/free-programming-books,Python,False,False,2013-10-11 06:50:37+00:00,2025-08-25 21:06:44+00:00,"[{'name': 'education', 'stars': 684}, {'name':...","{'name': 'education', 'stars': 684},{'name': '..."
4,public-apis/public-apis,public-apis,public-apis,363505,38166,4360,False,False,"[{'name': 'Python', 'size': 40479}, {'name': '...",2,...,None,public-apis/public-apis,public-apis/public-apis,Python,False,False,2016-03-20 23:49:42+00:00,2025-05-20 15:56:34+00:00,"[{'name': 'api', 'stars': 105389}, {'name': 'p...","{'name': 'api', 'stars': 105389},{'name': 'pub..."


In [ ]:
print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nStar distribution:")
print(df["stars"].describe())

In [ ]:
df["language"].value_counts().head(20).plot(kind="barh", figsize=(10, 6), title="Top 20 Languages")

In [ ]:
fe = FeatureExtractor()
features = fe.extract_all(df, query="python web framework")
print(f"Features shape: {features.shape}")
print(f"Features: {list(features.columns)}")
features.head()

In [ ]:
# Check which features correlate with stars (proxy for quality)
correlations = features.corrwith(df["stars"]).sort_values(ascending=False)
print("Feature correlations with stars:")
print(correlations)

## Baseline Ranking Model

Train a simple LightGBM ranker using topic-based synthetic relevance labels.

For each repo, relevance = topic overlap with a test query. This gives us ground truth for training without user data.

In [ ]:
# Create synthetic relevance labels based on topic overlap
# In production, these would come from user clicks/saves
test_queries = [
    "python machine learning",
    "javascript web framework", 
    "go microservices",
    "rust systems programming",
    "typescript react frontend",
]

# For each query, score repos by topic overlap
query = test_queries[0]
fe_query = FeatureExtractor()
relevance = fe_query.topic_overlap(df, query.split())
print(f"Query: '{query}'")
print(f"Repos with >0 overlap: {(relevance > 0).sum()}")

In [ ]:
from sklearn.model_selection import train_test_split

# Use features + synthetic relevance
X = features.fillna(0)
y = relevance.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Simple baseline: score = weighted combination of features
# DS would replace this with LightGBM/XGBoost later

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import ndcg_score

model = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
model.fit(X_train, y_train)

# Predict
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print(f"Train NDCG: {ndcg_score([y_train.values], [y_pred_train]):.4f}")
print(f"Test NDCG:  {ndcg_score([y_test.values], [y_pred_test]):.4f}")

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature Importance:")
print(importance)
importance.head(10).plot(kind="barh", figsize=(10, 4), title="Top 10 Feature Importances")

## Next Steps

- [ ] Try LightGBM ranker with `lambdarank` objective
- [ ] Add more features (readme analysis, contributor count, issue stats)
- [ ] Test with more queries and build a proper evaluation set
- [ ] Compare pretrained cross-encoder vs custom model
- [ ] Export trained model for production serving